In [ ]:
%run _bootstrap_dev.ipynb

## §2 — Portafoglio da analizzare

In [ ]:
#
# Quale portafoglio analizzare
#
portfolio = portfolio_alpha_euro
# portfolio = portfolio_alpha_sect
# portfolio = portfolio_alpha_world
# portfolio = portfolio_alpha_world_vanguard
# portfolio = portfolio_alpha_nasdaq100
# portfolio = portfolio_alpha_sp100
# portfolio = portfolio_germany_plan
# portfolio = portfolio_italy_big_cap
# portfolio = portfolio_alpha_quant # New Ema
# portfolio = portfolio_alpha_fact

# Profile di destinazione del PTF nel portafoglio del cliente
# Determina le soglie di overfitting check (S1-S4).
# "satellite": quota tattica, cerca alpha; "core": quota di base, prioritizza capital preservation.
profile = "satellite"   # "satellite" | "core"
pf_rot_cluster = None 
pf_rot_cluster_base = None

# --- Derivazione automatica dalle proprieta' del portfolio ---
# Flag calcolato PRIMA della risoluzione Wikipedia, quando tickers è ancora stringa.
survivorship_bias_universe = isinstance(portfolio['tickers'], str)
tickers = portfolio['tickers']
tickers = (
    extract_tickers_from_wikipedia(tickers, exclude=["GOOG"], rename={"BRK.B": "BRK-B"})
    if isinstance(tickers, str)
    else list(tickers)
)
benchmark_portfolio = portfolio['benchmark_portfolio']
benchmark_title     = portfolio['benchmark_title']
portfolio_title     = portfolio['Title']

# Percorso file WFO
wfo_results_dir = _TSLAB_DEV_R_WFO_RESULTS_DIR
wfo_file_save   = f"{wfo_results_dir}/{portfolio_title}_{year}.wfo_summary.csv"

# Date di analisi
start_date = "2015-01-01"
end_date   = None          # None = oggi
year       = 2026          # anno di selezione corrente

print(f"Portafoglio: {BOLD}{portfolio_title}{RESET}  |  Profile: {BOLD}{profile}{RESET}")
print(f"Ticker: {len(tickers)}  |  Benchmark: {benchmark_title}")
print(f"WFO file: {wfo_file_save}")
print(f"Survivorship bias universe: {survivorship_bias_universe}")

reports_dir = get_analysis_output_dir("r_analysis", ptf_name=portfolio_title.replace(' ', '_').lower())
plots_dir   = reports_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"reports_dir: {reports_dir}")
print(f"plots_dir: {plots_dir}")

## §3 — Download data

In [ ]:
init_cash      = 100_000
normalize      = False    # lasciare False nei rotazionali: NaN = titolo assente
lookback_buffer = 365

download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")

stocks_data, company_data = fetch_data_and_companies(
    tickers, download_start_date, end_date, normalize=normalize
)
stocks_data_raw = download_data(tickers, download_start_date, end_date, auto_adjust=False)

# Usato dalla Stability Analysis 
portfolio["stocks_data"] = stocks_data
portfolio["init_cash"]   = init_cash

if benchmark_portfolio:
    benchmark_data     = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max()).replace(0, np.nan).ffill()
    benchmark_data_raw = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max(),
                             auto_adjust=False).replace(0, np.nan).ffill()
elif benchmark_title:
    benchmark_data     = download_data(benchmark_title, stocks_data.index.min(), end_date)
    benchmark_data_raw = download_data(benchmark_title, stocks_data_raw.index.min(), end_date,
                                       auto_adjust=False)
else:
    benchmark_data = benchmark_data_raw = None

display(stocks_data)

In [ ]:
risk_off_tickers = portfolio.get('risk_off_tickers', risk_off_tickers)
risk_off_tickers_uniq = [
    t for t in risk_off_tickers
    if t not in tickers
]

# Tickers da usare in modalita risk_off (difensivi)
risk_off_data = download_data(risk_off_tickers_uniq, download_start_date, end_date) if risk_off_tickers else None
# workaround: forza sempre DataFrame
if isinstance(risk_off_data, pd.Series):
    risk_off_data = risk_off_data.to_frame()

## §4 — Stability Analysis

Valuta la coerenza dei flag binari della griglia su sotto-periodi storici.
Flag coerentemente negativi vengono fissati a `False` nella griglia ridotta
passata al WFO Standard.

> **Nota**: la stability si applica solo al path WFO Standard.
> WFO Clusterizzata (§5b) usa `build_cluster_grids` internamente e ignora la griglia esterna
> (debito metodologico documentato in CLAUDE.md).

In [ ]:
# Griglia completa — definita qui per essere condivisa tra stability e WFO
full_grid = {
    "rebalance_frequency":    ["QE", "ME"],
    "momentum_lookback_days": [10, 20, 40, 60],
    "riskparity_lookback_days": [10, 20, 40, 60],
    "n_top":                  resolve_n_top(portfolio.get('asset_type', 'stock'), profile),
    "use_acceleration":       [True, False],
    "momentum_weight":        [0.5, 0.7, 1.0],
    "filter_ema":             [True, False],
    "filter_volatility":      [True, False],
    "filter_min_momentum":    [True, False],
}

# Parametri WFO (condivisi tra Standard e pipeline interna Cluster)
ratio                  = "3:1"
metric                 = "Sharpe Ratio"
cores                  = -1
verbose                = False
force_next_year_params = True

n_full_trials = len(list(__import__('itertools').product(*full_grid.values())))
print(f"Full grid size: {n_full_trials} combinazioni")
print(f"n_top: {full_grid['n_top']}  (asset_type={portfolio.get('asset_type','stock')}, profile={profile})")

In [ ]:
if len(tickers) > 3:
    start_date_stability = stocks_data.dropna(how='all').index.min()
    print(f"Stability start effettivo: {start_date_stability.date()}")
    
    reduced_grid, stability_report = reduce_grid_via_stability(
        ptf_config       = portfolio,
        full_grid        = full_grid,
        full_start_date  = start_date_stability,
        full_end_date    = end_date,
        metric           = "CAGR",
        k                = 3,
        verbose          = True,
    )
    
    n_reduced_trials = len(list(__import__('itertools').product(*reduced_grid.values())))
    print(f"Reduced grid size: {n_reduced_trials} combinazioni ({n_full_trials - n_reduced_trials} eliminate)")
    
    # Salva stability report come CSV (audit trail)
    import os
    stability_report_path = reports_dir / f"{portfolio_title}_{year}_stability.csv"
    os.makedirs(os.path.dirname(stability_report_path), exist_ok=True)
    stability_report.to_csv(stability_report_path, index=False)
    print(f"Stability report salvato: {stability_report_path}")
    
    display(stability_report)
else:
    print(f"Universo troppo piccolo (n. tickers {len(tickers)}). Uso la griglia intera")
    reduced_grid =  full_grid

In [ ]:
# §4v2 — Griglia v2 e Stability Analysis (AGGIUNTIVA, non sostituisce §4 v1)
full_grid_v2 = {
    "rebalance_frequency":      ["QE", "ME"],
    "momentum_lookback_days":   [10, 20, 40, 60],
    "riskparity_lookback_days": [10, 20, 40, 60],
    "n_top":                    resolve_n_top(portfolio.get('asset_type', 'stock'), profile),
    "momentum_weight":          [0, 0.5, 1.0],
    "ivol_weight":              [0, 0.5, 1.0],
    "sortino_weight":           [0, 0.5, 1.0],
    "idio_weight":              [0, 0.5, 1.0],
    "filter_ema":               [True, False],
    "filter_volatility":        [True, False],
    "filter_min_momentum":      [True, False],
}

n_full_trials_v2 = len(list(__import__('itertools').product(*full_grid_v2.values())))
print(f"Full grid v2 size: {n_full_trials_v2} combinazioni")

if len(tickers) > 3:
    reduced_grid_v2, stability_report_v2 = reduce_grid_via_stability_v2(
        ptf_config        = portfolio,
        full_grid         = full_grid_v2,
        full_start_date   = start_date_stability,
        full_end_date     = end_date,
        metric            = "CAGR",
        k                 = 3,
        verbose           = True,
        benchmark_prices  = benchmark_data,   # nuovo, richiesto per idio_weight
    )

    n_reduced_trials_v2 = len(list(__import__('itertools').product(*reduced_grid_v2.values())))
    print(f"Reduced grid v2 size: {n_reduced_trials_v2} combinazioni ({n_full_trials_v2 - n_reduced_trials_v2} eliminate)")
    display(stability_report_v2)
else:
    print(f"Universo troppo piccolo. Uso la griglia v2 intera")
    reduced_grid_v2 = full_grid_v2

## §5 — Run WFO

### §5a — WFO Standard

Usa la `reduced_grid` prodotta dalla stability analysis.

In [ ]:
ratio_int = int(str(ratio).split(':')[0])
benchmark_start    = benchmark_data.dropna(how='all').index.min()
first_full_year    = pd.Timestamp(f"{benchmark_start.year + 1}-01-01")
pipeline_start_date = first_full_year - pd.DateOffset(years=ratio_int)

print(f"Benchmark start:         {benchmark_start.date()}")
print(f"Primo anno pieno comune: {first_full_year.date()}")
print(f"Pipeline start:          {pipeline_start_date.date()}")
print(f"Primo OOS atteso:        {first_full_year.date()}")


results_std = run_wfo_pipeline(
    # Dati
    stocks_data_raw        = stocks_data_raw,
    stocks_data            = stocks_data,
    benchmark_data         = benchmark_data,
    benchmark_data_raw     = benchmark_data_raw,
    tickers                = tickers,
    risk_off_data          = risk_off_data,
    # Parametri WFO
    ratio                  = ratio,
    metric                 = metric,
    start_date             = pipeline_start_date,
    end_date               = end_date,
    cores                  = cores,
    verbose                = verbose,
    force_next_year_params = force_next_year_params,
    use_clustering         = False,
    param_grid             = reduced_grid,   # stability-reduced
    # Parametri portafoglio
    portfolio_title        = portfolio_title,
    benchmark_title        = benchmark_title,
    init_cash              = init_cash,
    risk_on_off            = True,
    plot                   = True,
    profile                = profile,
    asset_type             = portfolio.get('asset_type', 'stock'),
    wfo_audit_path_base    = str(reports_dir / f"{portfolio_title}_{year}"),
)

# Accesso ai risultati
pf_rot_std           = results_std['pf_rot']           # con Risk ON/OFF
pf_rot_std_base      = results_std['pf_rot_base']      # senza Risk ON/OFF
regime               = results_std['regime']
summary_df_std       = results_std['summary_df']
sel_tickers_std      = results_std['sel_tickers']      # con Risk ON/OFF
sel_tickers_std_base = results_std['sel_tickers_base'] # senza Risk ON/OFF

# Verifica colonna universe (pre-condizione MC skill tests)
assert 'universe' in sel_tickers_std_base.columns, "Colonna 'universe' mancante in sel_tickers_std_base"
assert sel_tickers_std_base['universe'].apply(len).min() > 0
print(f"sel_tickers_std_base: {len(sel_tickers_std_base)} righe, universe OK")

In [ ]:
results_std_v2 = run_wfo_pipeline_v2(
    stocks_data_raw=stocks_data_raw, 
    stocks_data=stocks_data,
    benchmark_data=benchmark_data, 
    benchmark_data_raw=benchmark_data_raw,
    tickers=tickers, 
    risk_off_data=risk_off_data,
    ratio=ratio, 
    metric=metric, 
    start_date=pipeline_start_date, 
    end_date=end_date,
    cores=cores, 
    verbose=verbose, 
    # force_next_year_params=force_next_year_params,
    force_next_year_params=False,
    param_grid=reduced_grid_v2,
    portfolio_title=portfolio_title, 
    benchmark_title=benchmark_title,
    init_cash=init_cash, 
    risk_on_off=True, 
    plot=True,
    profile=profile, 
    asset_type=portfolio.get('asset_type', 'stock'),
    wfo_audit_path_base=str(reports_dir / f"{portfolio_title}_{year}"),
    benchmark_prices=benchmark_data_raw,   # ← aggiunto, richiesto per idio_weight>0
)

pf_rot_std_v2           = results_std_v2['pf_rot']
pf_rot_std_base_v2      = results_std_v2['pf_rot_base']
summary_df_std_v2       = results_std_v2['summary_df']
sel_tickers_std_v2      = results_std_v2['sel_tickers']
sel_tickers_std_base_v2 = results_std_v2['sel_tickers_base']

# Verifica colonna universe (stesso controllo di v1)
assert 'universe' in sel_tickers_std_base_v2.columns, "Colonna 'universe' mancante in sel_tickers_std_base_v2"
assert sel_tickers_std_base_v2['universe'].apply(len).min() > 0
print(f"sel_tickers_std_base_v2: {len(sel_tickers_std_base_v2)} righe, universe OK")

### §5b — WFO Clusterizzata

**Nota**: usa `build_cluster_grids` internamente — la `reduced_grid` della stability
non viene applicata. I flag binari variano liberamente per cluster
(debito metodologico, vedere CLAUDE.md).

In [ ]:
use_clustering     = True
adaptive_k         = True
adaptive_k_method  = 'hybrid'
max_clusters       = 5
lookback_days      = 504
n_top_min          = 2
save_plots         = True

results_cluster = run_wfo_pipeline(
    # Dati
    stocks_data_raw    = stocks_data_raw,
    stocks_data        = stocks_data,
    benchmark_data     = benchmark_data,
    benchmark_data_raw = benchmark_data_raw,
    tickers            = tickers,
    risk_off_data      = risk_off_data,
    # Parametri WFO
    ratio              = ratio,
    metric             = metric,
    start_date         = pipeline_start_date,
    end_date           = end_date,
    cores              = cores,
    verbose            = verbose,
    force_next_year_params = force_next_year_params,
    # Parametri clustering
    use_clustering     = use_clustering,
    adaptive_k         = adaptive_k,
    adaptive_k_method  = adaptive_k_method,
    n_clusters         = max_clusters,
    lookback_days      = lookback_days,
    n_top_min          = n_top_min,
    # Parametri portafoglio
    portfolio_title    = portfolio_title,
    benchmark_title    = benchmark_title,
    init_cash          = init_cash,
    risk_on_off        = True,
    plot               = True,
    # Salvataggio plot
    save_plots         = save_plots,
    plots_dir          = plots_dir,
    profile            = profile,
    asset_type         = portfolio.get('asset_type', 'stock'),
    wfo_audit_path_base = str(reports_dir / f"{portfolio_title}_{year}"),
)

pf_rot_cluster           = results_cluster['pf_rot']
pf_rot_cluster_base      = results_cluster['pf_rot_base']
regime_cluster           = results_cluster['regime']
summary_df_cluster       = results_cluster['summary_df']
sel_tickers_cluster      = results_cluster['sel_tickers']
sel_tickers_cluster_base = results_cluster['sel_tickers_base']

# results_cluster = None   # placeholder se non eseguita

### §5c — Confronto WFO

Confronto metriche di performance Standard vs Clusterizzata.
Il confronto dei verdetti OFC (S1-S4) è in §6.

In [ ]:
metrics_df = compare_wfo_pipelines(
    results = {
        "v1": results_std,
        "v2": results_std_v2,
    },
    portfolio_title = portfolio_title,
    benchmark_title = benchmark_title,
    plot_radar      = True,
    save_plots      = True,
    plots_dir       = plots_dir,
)
display(metrics_df)

In [ ]:
def quick_sanity_check(pf_rot, pf_rot_base=None, label="Portfolio", min_flags_to_fail=2):
    """
    Controllo qualitativo veloce su un Portfolio VBT, prima di spendere
    tempo su OFC/MC. Usa solo pf.stats(), nessun bootstrap aggiuntivo.
    Segnala soglie d'allarme empiriche — non sono verità statistica,
    solo euristiche per decidere se vale la pena procedere oltre.

    Returns
    -------
    dict con:
        'proceed' : bool  — True se consigliato procedere a OFC/MC
        'flags'   : list[str] — segnali d'allarme raccolti
        'n_flags' : int

    Verdetto: 'proceed' = False se il numero di segnali d'allarme
    raggiunge o supera min_flags_to_fail (default 2). Soglia empirica,
    non statisticamente derivata — pensata come euristica di pre-filtro,
    non come sostituto di OFC/MC.
    """
    print(f"\n{'='*60}")
    print(f"  Quick Sanity Check — {label}")
    print(f"{'='*60}")

    flags = []

    def _check(pf, sublabel):
        if pf is None:
            print(f"  [{sublabel}] non disponibile, skip")
            return
        s = pf.stats()

        total_trades = s.get('Total Trades', float('nan'))
        win_rate     = s.get('Win Rate [%]', float('nan'))
        profit_factor= s.get('Profit Factor', float('nan'))
        expectancy   = s.get('Expectancy', float('nan'))
        worst_trade  = s.get('Worst Trade [%]', float('nan'))
        best_trade   = s.get('Best Trade [%]', float('nan'))
        sharpe       = s.get('Sharpe Ratio', float('nan'))
        max_dd       = s.get('Max Drawdown [%]', float('nan'))

        print(f"\n  --- {sublabel} ---")
        print(f"  Total Trades     : {total_trades}")
        print(f"  Win Rate [%]     : {win_rate:.2f}")
        print(f"  Profit Factor    : {profit_factor:.3f}")
        print(f"  Expectancy       : {expectancy:.4f}")
        print(f"  Sharpe Ratio     : {sharpe:.3f}")
        print(f"  Max Drawdown [%] : {max_dd:.2f}")
        print(f"  Best/Worst Trade : {best_trade:.2f}% / {worst_trade:.2f}%")

        if profit_factor < 1.0:
            flags.append(f"[{sublabel}] Profit Factor < 1.0 ({profit_factor:.3f}) — perdite superano i guadagni in valore assoluto")
        if expectancy < 0:
            flags.append(f"[{sublabel}] Expectancy negativa ({expectancy:.4f}) — il sistema perde in media per trade")
        if sharpe < 0.3:
            flags.append(f"[{sublabel}] Sharpe Ratio basso ({sharpe:.3f}) — rendimento risk-adjusted debole")
        if best_trade and abs(worst_trade) > 3 * abs(best_trade):
            flags.append(f"[{sublabel}] Worst Trade ({worst_trade:.2f}%) >> Best Trade ({best_trade:.2f}%) — possibile outlier di coda pesante")
        if not pd.isna(total_trades) and not pd.isna(win_rate):
            if win_rate > 55 and profit_factor < 1.05:
                flags.append(f"[{sublabel}] Win Rate alto ({win_rate:.1f}%) ma Profit Factor vicino/sotto 1 — pattern 'tanti piccoli vincenti, poche grandi perdite'")

    _check(pf_rot,      "Risk ON/OFF")
    _check(pf_rot_base, "Base")

    proceed = len(flags) < min_flags_to_fail

    print(f"\n{'-'*60}")
    if flags:
        print(f"  ⚠ {len(flags)} segnali d'allarme:")
        for f in flags:
            print(f"    - {f}")
    else:
        print(f"  ✓ Nessun segnale d'allarme evidente.")

    print(f"\n  VERDETTO: {'✓ PROCEDI con OFC/MC' if proceed else '✗ SCONSIGLIATO procedere — rivedere turnover/griglia prima'}")
    print(f"  (soglia: {min_flags_to_fail}+ segnali = sconsigliato; trovati: {len(flags)})")
    print(f"{'='*60}\n")

    return {
        'proceed': proceed,
        'flags':   flags,
        'n_flags': len(flags),
    }
check_v1 = quick_sanity_check(pf_rot_std, pf_rot_std_base, label="Alpha World — v1")
check_v2 = quick_sanity_check(pf_rot_std_v2, pf_rot_std_base_v2, label="Alpha World — v2")

# Riusabile programmaticamente, es. per skip condizionale:
if not check_v2['proceed']:
    print("Salto OFC/MC per v2 (troppi segnali d'allarme)")

In [ ]:
# ── Configurazione: una entry per ciascun motore/risultato da testare ──
wfo_runs = {
    "v1": dict(
        summary_df       = summary_df_std,
        param_grid        = reduced_grid,
        n_total_trials    = n_full_trials,
        stability_report  = stability_report,
        pf_rot            = pf_rot_std,
        pf_rot_base       = pf_rot_std_base,
        sel_tickers       = sel_tickers_std,
        sel_tickers_base  = sel_tickers_std_base,
        regime            = regime,
    ),
    "v2": dict(
        summary_df        = summary_df_std_v2,
        param_grid         = reduced_grid_v2,
        n_total_trials     = n_full_trials_v2,
        stability_report   = stability_report_v2,
        pf_rot             = pf_rot_std_v2,
        pf_rot_base        = pf_rot_std_base_v2,
        sel_tickers        = sel_tickers_std_v2,
        sel_tickers_base   = sel_tickers_std_base_v2,
        regime             = None,
    ),
}

results_pipeline = {}

for name, cfg in wfo_runs.items():
    print(f"\n{'#'*60}\n  {name}\n{'#'*60}")

    # 1) Sanity check
    check = quick_sanity_check(cfg['pf_rot'], cfg['pf_rot_base'], label=name)

    if not check['proceed']:
        print(f"  [{name}] SKIP OFC/MC — sanity check fallito ({check['n_flags']} flag)")
        results_pipeline[name] = dict(sanity=check, ofc=None, mc=None)
        continue

    # 2) OFC
    ofc_passed, ofc_report = overfitting_check_rotational(
        wfo_summary       = cfg['summary_df'],
        stocks_data       = stocks_data,
        benchmark_data    = benchmark_data,
        param_grid        = cfg['param_grid'],
        profile           = profile,
        n_total_trials    = cfg['n_total_trials'],
        stability_report  = cfg['stability_report'],
        seed              = 42,
        verbose           = True,
    )

    # 3) MC (solo se OFC passa — risparmia tempo se già bocciato; rimuovi
    #    l'if se vuoi MC sempre, indipendentemente da OFC)
    mc_out = None
    if ofc_passed:
        plots_dir_run = Path(plots_dir) / name
        plots_dir_run.mkdir(parents=True, exist_ok=True)
        ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
            pf_rot                 = cfg['pf_rot'],
            pf_rot_base            = cfg['pf_rot_base'],
            regime                 = cfg['regime'],
            sel_tickers            = cfg['sel_tickers'],
            sel_tickers_base       = cfg['sel_tickers_base'],
            stocks_data            = stocks_data,
            benchmark_data         = benchmark_data,
            tickers_master         = tickers,
            init_cash               = init_cash,
            n_simulations            = 1000,
            seed                     = 42,
            block_size               = 10,
            vol_window               = 60,
            n_vol_quantiles          = 3,
            show_method_plots        = True,
            show_method_summaries    = True,
            save_plots               = True,
            plots_dir                = plots_dir_run,
        )
        mc_out = dict(ci_results=ci_results, ci_summary_df=ci_summary_df,
                      skill_results=skill_results, skill_summary_df=skill_summary_df)
    else:
        print(f"  [{name}] SKIP MC — OFC non promosso")

    results_pipeline[name] = dict(sanity=check, ofc=dict(passed=ofc_passed, report=ofc_report), mc=mc_out)

# ── Riepilogo finale ──
print(f"\n{'='*60}\n  RIEPILOGO\n{'='*60}")
for name, r in results_pipeline.items():
    sanity_ok = r['sanity']['proceed']
    ofc_ok    = r['ofc']['passed'] if r['ofc'] else None
    mc_done   = r['mc'] is not None
    print(f"  {name:6s} | sanity={sanity_ok} | ofc={ofc_ok} | mc_eseguito={mc_done}")

In [ ]:
# print(results_pipeline['v1']['ofc']['report'].keys())

In [ ]:
for name in ["v1", "v2"]:
    signals = results_pipeline[name]['ofc']['report']['signals']
    print(f"\n{'='*60}\n  {name} — dettaglio segnali OFC\n{'='*60}")
    for sig_name, sig_data in signals.items():
        print(f"  {sig_name}: {sig_data}")

## §6 — Overfitting Check

Valuta 4 segnali di overfitting (S1 plateau, S2 coerenza flag,
S3 cross-sectional skill, S4 DSR). Verdetti separati per Standard e Cluster.
Vedi CLAUDE.md "Interpretation framework: Reshuffle vs S3".

In [ ]:
# import json
# import itertools
# from pathlib import Path

# # --- OFC Standard ---
# ofc_passed_std, ofc_report_std = overfitting_check_rotational(
#     wfo_summary      = summary_df_std,
#     stocks_data      = stocks_data,
#     benchmark_data   = benchmark_data,
#     param_grid       = reduced_grid,
#     profile          = profile,
#     n_total_trials   = n_full_trials,     # penalizzazione conservativa: full grid
#     stability_report = stability_report,
#     seed             = 42,
#     verbose          = True,
# )

# # Salva ofc_report come JSON (audit trail; caricabile in §10 senza re-run)
# ofc_report_path = reports_dir / f"{portfolio_title}_{year}_ofc_std.json"
# Path(ofc_report_path).parent.mkdir(parents=True, exist_ok=True)
# with open(ofc_report_path, "w") as f:
#     json.dump(ofc_report_std, f, default=str, indent=2)
# print(f"OFC report std salvato: {ofc_report_path}")
# print(f"OFC Standard — promoted={ofc_passed_std}")

In [ ]:
# # --- OFC v2 ---
# ofc_passed_std_v2, ofc_report_std_v2 = overfitting_check_rotational(
#     wfo_summary      = summary_df_std_v2,
#     stocks_data      = stocks_data,
#     benchmark_data   = benchmark_data,
#     param_grid       = reduced_grid_v2,
#     profile          = profile,
#     n_total_trials   = n_full_trials_v2,
#     stability_report = stability_report_v2,
#     seed             = 42,
#     verbose          = True,
# )
# ofc_report_path_v2 = reports_dir / f"{portfolio_title}_{year}_ofc_std_v2.json"
# Path(ofc_report_path_v2).parent.mkdir(parents=True, exist_ok=True)
# with open(ofc_report_path_v2, "w") as f:
#     json.dump(ofc_report_std_v2, f, default=str, indent=2)
# print(f"OFC report std v2 salvato: {ofc_report_path_v2}")
# print(f"OFC Standard v2 — promoted={ofc_passed_std_v2}")

In [ ]:
# # --- OFC Cluster (solo se WFO Clusterizzata è stata eseguita) ---
# if 'results_cluster' in locals() and results_cluster is not None:
#     ofc_passed_cluster, ofc_report_cluster = overfitting_check_rotational(
#         wfo_summary      = summary_df_cluster,
#         stocks_data      = stocks_data,
#         benchmark_data   = benchmark_data,
#         param_grid       = full_grid,       # cluster usa full_grid (no stability)
#         profile          = profile,
#         n_total_trials   = n_full_trials,
#         seed             = 42,
#         verbose          = True,
#     )
#     ofc_report_cluster_path = reports_dir / f"{portfolio_title}_{year}_ofc_cluster.json"
#     Path(ofc_report_cluster_path).parent.mkdir(parents=True, exist_ok=True)
#     with open(ofc_report_cluster_path, "w") as f:
#         json.dump(ofc_report_cluster, f, default=str, indent=2)
#     print(f"OFC report cluster salvato: {ofc_report_cluster_path}")
#     print(f"OFC Cluster — promoted={ofc_passed_cluster}")

#     # Tabella comparativa OFC Standard vs Cluster
#     print("\n=== Confronto OFC Standard vs Cluster ===")
#     for sig in ['s1_plateau', 's2_coherence', 's3_random_selection', 's4_dsr']:
#         v_std = ofc_report_std.get(sig, {}).get('passed', '?')
#         v_clu = ofc_report_cluster.get(sig, {}).get('passed', '?')
#         print(f"  {sig:25s}  STD={'PASS' if v_std else 'FAIL'}  CLU={'PASS' if v_clu else 'FAIL'}")
#     print(f"  {'Overall':25s}  STD={'PASS' if ofc_passed_std else 'FAIL'}  "
#           f"CLU={'PASS' if ofc_passed_cluster else 'FAIL'}")
# else:
#     ofc_passed_cluster = None
#     ofc_report_cluster = None
#     print("WFO Clusterizzata non eseguita — OFC Cluster non disponibile.")

## §7 — MC (Monte Carlo Validation)

In [ ]:

# from pathlib import Path

# # Sub-directory per i plot dei due path (evita sovrascrittura)
# plots_dir_std     = Path(plots_dir) / 'std'
# plots_dir_cluster = Path(plots_dir) / 'cluster'
# plots_dir_std_v2  = Path(plots_dir) / 'std_v2'

# plots_dir_std.mkdir(parents=True, exist_ok=True)
# plots_dir_cluster.mkdir(parents=True, exist_ok=True)
# plots_dir_std_v2.mkdir(parents=True, exist_ok=True)


# # Run completa (n=1000) con grafici — Path Standard v1
# ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
#     pf_rot              = pf_rot_std,
#     pf_rot_base         = pf_rot_std_base,
#     regime              = regime,
#     sel_tickers         = sel_tickers_std,
#     sel_tickers_base    = sel_tickers_std_base,
#     stocks_data         = stocks_data,
#     benchmark_data      = benchmark_data,
#     tickers_master      = tickers,
#     init_cash           = init_cash,
#     n_simulations       = 1000,
#     seed                = 42,
#     block_size          = 10,
#     vol_window          = 60,
#     n_vol_quantiles     = 3,
#     show_method_plots   = True,
#     show_method_summaries = True,
#     save_plots            = True,
#     plots_dir             = plots_dir_std,   # ← sub-dir Standard
# )

# # Run completa (n=1000) con grafici — Path Standard v2
# ci_results_v2, ci_summary_df_v2, skill_results_v2, skill_summary_df_v2 = run_all_mc_methods_rotational(
#     pf_rot                = pf_rot_std_v2,
#     pf_rot_base           = pf_rot_std_base_v2,
#     regime                = None,   # results_std_v2['regime'] è sempre None nel path v2/Path B
#     sel_tickers            = sel_tickers_std_v2,
#     sel_tickers_base       = sel_tickers_std_base_v2,
#     stocks_data            = stocks_data,
#     benchmark_data          = benchmark_data,
#     tickers_master          = tickers,
#     init_cash               = init_cash,
#     n_simulations           = 1000,
#     seed                    = 42,
#     block_size              = 10,
#     vol_window              = 60,
#     n_vol_quantiles         = 3,
#     show_method_plots       = True,
#     show_method_summaries   = True,
#     save_plots              = True,
#     plots_dir               = plots_dir_std_v2,
# )

# # Run completa (n=1000) con grafici — Path Cluster
# if pf_rot_cluster is not None and pf_rot_cluster_base is not None:
#     ci_results_cluster, ci_summary_df_cluster, skill_results_cluster, skill_summary_df_cluster = run_all_mc_methods_rotational(
#         pf_rot              = pf_rot_cluster,
#         pf_rot_base         = pf_rot_cluster_base,
#         regime              = regime_cluster,
#         sel_tickers         = sel_tickers_cluster,
#         sel_tickers_base    = sel_tickers_cluster_base,
#         stocks_data         = stocks_data,
#         benchmark_data      = benchmark_data,
#         tickers_master      = tickers,
#         init_cash           = init_cash,
#         n_simulations       = 1000,
#         seed                = 42,
#         block_size          = 10,
#         vol_window          = 60,
#         n_vol_quantiles     = 3,
#         show_method_plots   = True,
#         show_method_summaries = True,
#         save_plots            = True,
#         plots_dir             = plots_dir_cluster,   # ← sub-dir Cluster
#     )
# else:
#     ci_results_cluster        = None
#     ci_summary_df_cluster     = None
#     skill_results_cluster     = None
#     skill_summary_df_cluster  = None

In [ ]:
# display(ci_summary_df)
# display(skill_summary_df)
# display(ci_summary_df_cluster)
# display(skill_summary_df_cluster)

## §8 — Decisione finale + Scheda PTF

Aggrega tutti i segnali in una tabella decisionale per path.
L'ultima riga "User decision" è compilata a mano dal modellista.

In [ ]:
# STEP 8 — Decisione finale + Scheda PTF + Relazione Tecnica

from datetime import date
_today_iso = date.today().isoformat()

# Output paths — stessa struttura e naming del CLI (no sottocartelle extra,
# no doppio livello data: reports_dir è già <root>/outputs/reports/<ptf>/<data>/)
_ptf_name = portfolio_title.replace(' ', '_').lower()
_card_path = reports_dir / f"{_ptf_name}_{year}_{profile}.md"
_pdf_path  = reports_dir / f"{portfolio_title}_{year}_{profile}_Relazione_Tecnica.pdf"
_card_path.parent.mkdir(parents=True, exist_ok=True)
_pdf_path.parent.mkdir(parents=True, exist_ok=True)



skill_profile_std, skill_profile_cluster = compute_skill_profile(
    mc_skill=skill_results,
    mc_skill_cluster=skill_results_cluster,
)

# Compat: per ora il parametro skill_profile passato a generate_relazione_tecnica 
# resta singolo. Useremo skill_profile_std come default; lo sdoppiamento avverrà 
# nella firma di generate_relazione_tecnica al pezzo 4.
skill_profile = skill_profile_std

_wfo_config = {
    'ratio':            ratio,
    'metric':           metric,
    'n_full_trials':    n_full_trials,
    'n_reduced_trials': n_reduced_trials,
    'wfo_file_save':    wfo_file_save,
    'use_clustering':   use_clustering,
    'n_bootstrap_ofc':  1000,
    'n_bootstrap_mc':   1000,
}

_cluster_result = (
    results_cluster.get('cluster_result') if results_cluster else None
)

_metrics_comparison = {
    'cluster_riskoff': results_cluster.get('pf_rot')      if results_cluster else None,
    'cluster_base':    results_cluster.get('pf_rot_base') if results_cluster else None,
    'std_riskoff':     results_std.get('pf_rot'),
    'std_base':        results_std.get('pf_rot_base'),
    'benchmark':       results_std.get('pf_benchmark') or results_std.get('pf_benchmark_base'),
}


# 1. Stampa DECISIONE FINALE a video
print_final_decision(
    portfolio_title    = portfolio_title,
    year               = year,
    profile            = profile,
    ofc_report_std     = ofc_report_std,
    ofc_report_cluster = ofc_report_cluster,
    mc_skill           = skill_results,
    mc_ci              = ci_summary_df,
    skill_profile      = skill_profile,
    mc_skill_cluster   = skill_results_cluster,     # NEW
    mc_ci_cluster      = ci_summary_df_cluster,     # NEW
)

# 2. Scrivi PTF Card markdown
generate_ptf_card_md(
    portfolio_title    = portfolio_title,
    year               = year,
    profile            = profile,
    benchmark          = benchmark_title,
    period             = (str(pipeline_start_date), _today_iso),
    universe_size      = len(tickers),
    wfo_config         = _wfo_config,
    cluster_result     = _cluster_result,
    metrics_comparison = _metrics_comparison,
    ofc_report_std     = ofc_report_std,
    ofc_report_cluster = ofc_report_cluster,
    mc_skill           = skill_results,
    mc_ci              = ci_summary_df,
    mc_skill_cluster = skill_results_cluster,        # NEW
    mc_ci_cluster    = ci_summary_df_cluster,        # NEW
    skill_profile      = skill_profile,
    output_path        = _card_path,
)
print(f"PTF Card MD: {_card_path}")

# 3. Genera scheda tecnica PDF
generate_relazione_tecnica(
    portfolio_title              = portfolio_title,
    year                         = year,
    profile                      = profile,
    benchmark                    = benchmark_title,
    period                       = (str(pipeline_start_date), _today_iso),
    universe_size                = len(tickers),
    wfo_config                   = _wfo_config,
    cluster_result               = _cluster_result,
    metrics_comparison           = _metrics_comparison,
    ofc_report_std               = ofc_report_std,
    ofc_report_cluster           = ofc_report_cluster,
    mc_skill                     = skill_results,
    mc_ci                        = ci_summary_df,
    skill_profile                = skill_profile_std,            # B-005: ex skill_profile
    skill_profile_cluster        = skill_profile_cluster,        # B-005: nuovo
    plots_dir                    = plots_dir,
    output_path                  = _pdf_path,
    ci_results                   = ci_results,
    mc_skill_cluster             = skill_results_cluster,        # passato come prima
    mc_ci_cluster                = ci_summary_df_cluster,        # passato come prima
    survivorship_bias_universe   = survivorship_bias_universe,
)

print(f"Relazione tecnica PDF: {_pdf_path}")

In [ ]:
# §8 — Salvataggio WFO per il RUNTIME (path scelto dall'architetto)
#
# fix/r-runtime-save-chosen-path:
# Questa cella scrive su `wfo_file_save` — l'UNICO file letto da r_run_portfolio
# nel runtime (via collect_selections_from_summary +
# extract_operational_params_from_summary) — il summary_df del path
# EFFETTIVAMENTE scelto qui in §8.
#
# Compilare `path_scelto` a mano DOPO aver letto la relazione tecnica PDF.

path_scelto = "CLUSTER"   # "STANDARD" oppure "CLUSTER" — compilare a mano dopo lettura PDF

_path_scelto = str(path_scelto).strip().upper()
if _path_scelto not in ("STANDARD", "CLUSTER"):
    raise ValueError(
        f"path_scelto non valido: '{path_scelto}'. Valori attesi: 'STANDARD' o 'CLUSTER'."
    )

if _path_scelto == "CLUSTER":
    if results_cluster is None:
        raise RuntimeError(
            "path_scelto='CLUSTER' ma results_cluster is None: il path Cluster non e' "
            "stato eseguito (§5b — WFO Clusterizzata). Esegui il WFO Cluster prima di "
            "salvare, oppure imposta path_scelto='STANDARD'."
        )
    _summary_df_runtime = summary_df_cluster
    _param_grid_runtime = full_grid       # il path Cluster usa full_grid (no stability)
else:
    _summary_df_runtime = summary_df_std
    _param_grid_runtime = reduced_grid    # il path Standard usa la reduced_grid (stability)

save_rotational_wfo_summary(
    summary_df             = _summary_df_runtime,
    start_date             = start_date,
    end_date               = end_date,
    file_path              = wfo_file_save,
    param_grid             = _param_grid_runtime,
    metric                 = metric,
    ratio                  = ratio,
    force_next_year_params = force_next_year_params,
    extra_meta             = {"deployed_path": _path_scelto},
)
print(f"[§8] Path scelto per il RUNTIME: {_path_scelto}")
print(f"[§8] WFO summary salvato su (letto da r_run_portfolio): {wfo_file_save}")

## §9 — Performance

In [ ]:
# Ci sono 4 portafogli disponibili:
# pf_rot_std_base        (WFO non clusterizzata)
# pf_rot_std             (WFO non clusterizzata, Risk ON/OFF)
# pf_rot_cluster_base    (WFO clusterizzata)
# pf_rot_cluster         (WFO clusterizzata, Risk ON/OFF)

# E per ognuno di essi 2 possibili confronto con altrettanti benchmark
# Benchmark esterno: definito dal portafoglio (benchmark_data=benchmark_data)
# Benchmark interno: B&H intero universo  (benchmark_data=None)

# Schema decisionale rapido                                                                                                      
                                                                                                                             
# Universo piccolo/omogeneo?
#   ├─ Sì → std_base  (o std se vuoi protezione drawdown)                                                                        
#   └─ No (grande/eterogeneo) → cluster_base  (o cluster se vuoi regime switching)                                               
                                                                                                                             
# Vuoi ridurre il max drawdown?                                                                                                  
#   └─ Sì → aggiungi Risk ON/OFF (std → std_on, cluster → cluster_on)                                                            
                                                                                                                             
# Periodo OOS corto (<3 anni)?
#   └─ Preferisci la versione _base: meno parametri, più robusta                                                                 
                                                                                                                             
# ---
# In pratica, pf_rot_cluster è il più potente ma anche il più fragile se i dati sono pochi. pf_rot_std_base è il più stabile e   
# interpretabile. Gli altri due stanno nel mezzo.                                                                                

pf = pf_rot_cluster
sel_tickers = sel_tickers_cluster

out = generate_rotational_portfolio_performance(
    pf=pf,
    portfolio_title=portfolio_title,
    sel_tickers=sel_tickers,
    benchmark=benchmark_title,
    benchmark_data=benchmark_data,   # prezzi close
    alpha_analysis=True,
    show_plots=True,
    plot_start_date='2026-05-11'
)

In [ ]:
# Engine Sanity Check
health_df, ticker_df, selection_log, details = build_engine_health_check(
    pf_rot_std,
    sel_tickers_std,
    prices    = stocks_data,
    start_date = start_date,
    end_date   = end_date,
    include_prev = True,
)
my_display(health_df, "Health check motore rotazionale")
my_display(selection_log, "Audit selezioni")

## §10 — Load WFO Results

Riprende un'analisi precedente senza rieseguire WFO, stability e OFC.
Richiede che §4 abbia salvato `stability_report.csv` e §6 abbia salvato `ofc_report.json`.

In [ ]:
# Carica wfo_summary, stability_report e ofc_report da run precedente
import json
import os

summary_df       = load_wfo_summary(wfo_file_save)

stability_report_path   = reports_dir / f"{portfolio_title}_{year}_stability.csv"
ofc_report_std_path     = reports_dir / f"{portfolio_title}_{year}_ofc_std.json"

if os.path.exists(stability_report_path):
    stability_report = pd.read_csv(stability_report_path)
    print(f"Stability report caricato: {stability_report_path}")
else:
    stability_report = None
    print(f"[WARN] stability report non trovato: {stability_report_path}")

if os.path.exists(ofc_report_std_path):
    with open(ofc_report_std_path) as f:
        ofc_report_std = json.load(f)
    print(f"OFC report std caricato: {ofc_report_std_path}")
else:
    ofc_report_std = None
    print(f"[WARN] OFC report std non trovato: {ofc_report_std_path}")

display(summary_df)

In [ ]:
# Ricostruisce pf_rot e sel_tickers dalla wfo_summary caricata
# Sostituisce la chiamata stale build_rotational_portfolios_from_wfo_result
pf_rot, pf_bh, sel_tickers = build_portfolio_from_wfo_summary(
    summary_df     = summary_df,
    stocks_data    = stocks_data,
    benchmark_data = benchmark_data,
    start_date     = start_date,
    end_date       = end_date,
    benchmark_title = benchmark_title,
    portfolio_name  = portfolio_title,
    init_cash       = init_cash,
    plot            = True,
    show_report     = True,
)

In [ ]:
# Ricostruisce pf_rot e sel_tickers dalla wfo_summary caricata
# Sostituisce la chiamata stale build_rotational_portfolios_from_wfo_result
pf_rot, pf_bh, sel_tickers = build_portfolio_from_wfo_summary(
    summary_df     = summary_df_cluster,
    stocks_data    = stocks_data,
    benchmark_data = benchmark_data,
    start_date     = start_date,
    end_date       = end_date,
    benchmark_title = benchmark_title,
    portfolio_name  = portfolio_title,
    init_cash       = init_cash,
    plot            = True,
    show_report     = True,
)

In [ ]:
# Dopo il load e' possibile eseguire §8 (Decisione finale) ricalcolando
# le variabili derivate da ofc_report_std e skill_results se disponibili.
# Oppure ri-eseguire §7 (MC) senza rieseguire il WFO.
print("Load completato. Variabili disponibili:")
print(f"  summary_df:       {summary_df.shape}")
print(f"  pf_rot:           {type(pf_rot).__name__}")
print(f"  sel_tickers:      {sel_tickers.shape}")
print(f"  stability_report: {'OK' if stability_report is not None else 'non disponibile'}")
print(f"  ofc_report_std:   {'OK' if ofc_report_std is not None else 'non disponibile'}")

## Snapshot

In [ ]:
# # Snapshot pre-fix MC — portfolio_germany_plan
# import shutil, json
# from pathlib import Path
# from datetime import datetime

# snap_dir = Path(f"../dev/snapshots/pre_mc_fix/{portfolio_title}").resolve()
# snap_dir.mkdir(parents=True, exist_ok=True)

# # 1. Re-cattura output testuale di print_final_decision
# import io, contextlib
# buf = io.StringIO()
# with contextlib.redirect_stdout(buf):
#     print_final_decision(
#         portfolio_title    = portfolio_title,
#         year               = year,
#         profile            = profile,
#         ofc_report_std     = ofc_report_std,
#         ofc_report_cluster = ofc_report_cluster,
#         mc_skill           = skill_results,
#         mc_ci              = ci_summary_df,
#         skill_profile      = skill_profile,
#     )
# (snap_dir / "print_final_decision.txt").write_text(buf.getvalue())

# # 2. Copia PTF Card
# ptf_card_src = Path(_card_path).resolve()
# if ptf_card_src.exists():
#     shutil.copy(ptf_card_src, snap_dir / "ptf_card.md")

# # 3. Copia scheda tecnica PDF
# pdf_src = Path(_pdf_path).resolve()
# if pdf_src.exists():
#     shutil.copy(pdf_src, snap_dir / "scheda_tecnica.pdf")

# # 4. Metadati snapshot
# metadata = {
#     "portfolio_title": portfolio_title,
#     "snapshot_date": datetime.now().isoformat(),
#     "branch": "fix/r-mc-cluster-symmetry",
#     "bug_status": "PRESENT (pre-fix baseline)",
#     "files": {
#         "print_final_decision.txt": (snap_dir / "print_final_decision.txt").exists(),
#         "ptf_card.md": (snap_dir / "ptf_card.md").exists(),
#         "scheda_tecnica.pdf": (snap_dir / "scheda_tecnica.pdf").exists(),
#     },
# }
# (snap_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

# print(f"Snapshot salvato: {snap_dir}")
# for k, v in metadata["files"].items():
#     print(f"  {k}: {'OK' if v else 'MISSING'}")

In [ ]:
import importlib
import r_functions
importlib.reload(r_functions)
from r_functions import build_cluster_grids
import inspect
print(inspect.signature(build_cluster_grids))